# Exercise 4 | TKO_7092 Evaluation of Machine Learning Methods
---

Student name: Talha Rizwan<br>
Student number: 2509714<br>
Student email: tarizw@utu.fi<br>

---

The deadline for returning this exercise is **25.2.2026**.

If you have any questions about this exercise, please contact Riikka Numminen (rimanu@utu.fi) in good time before the deadline.

## Nested cross-validation for feature selection
In this exercise, the task is to use leave-one-out cross-validation for model selection to understand the effect of the winner's curse. 
This is demonstrated by using greedy forward selection and a random binary data set.
The data set is a balanced sample of size 60 (i.e. 30 positives and 30 negatives) with a hundred features. The data are i.i.d., and every feature follows a Bernoulli distribution with $p=0.5$. Thus, there is no signal in the data. 

The model to be used is 1-nearest neighbour with 10 features, and the greedy forward selection is used to select the best 10 features among all the features.
Leave-one-out cross-validation is used for performance evaluation, and the prediction performance is measured as accuracy. 

### Greedy forward feature selection
Greedy forward feature selection is an iterative feature selection process, where the features are selected one by one, avoiding a need to iterate through every possible combination of features. The features are selected as follows:
- First, every feature is tested solely and the best is selected.
- Then the selected feature is tested together with any other remaining feature and the best such a set of two features is selected.
- Then that set of the selected two features is tested together with any other remaining feature and the best set of three features is selected.
- The process is then continued accordingly until the desired amount of features is selected.

### Implement the following tasks to complete this exercise:
1. Use leave-one-out cross-validation to select the best 10 features. Report the optimal set of features and the corresponding accuracy.
2. Use nested leave-one-out cross-validation (leave-one-out on both layers of cross-validation) to obtain an estimate of the prediction accuracy on unseen data, when the final hypotheses are obtained according to the procedure in the first step.
3. Explain the difference in the obtained accuracies.

### Import libraries

In [12]:
# In this cell, import all the libraries that you will use in this notebook.
import numpy as np

### Load the data
The labels are saved in a file *y_generated.csv*, and the features in file *X_generated.csv*.

In [13]:
# Read the data files. Verify that the data dimensions are as expected.
X = np.loadtxt('X_generated.csv', delimiter=',', dtype=int)
y = np.loadtxt('y_generated.csv', delimiter=',', dtype=int)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class counts (0,1):', np.bincount(y))

X shape: (60, 100)
y shape: (60,)
Class counts (0,1): [30 30]


### Leave-one-out cross-validation

In [14]:
# Part 1: Leave-one-out cross-validation for greedy forward feature selection
def loocv_accuracy_1nn(X_data, y_data, feature_idx):
    X_sub = X_data[:, feature_idx]
    dists = np.sum(X_sub[:, None, :] != X_sub[None, :, :], axis=2).astype(float)
    np.fill_diagonal(dists, np.inf)
    nn_idx = np.argmin(dists, axis=1)
    y_pred = y_data[nn_idx]
    return np.mean(y_pred == y_data)

def greedy_forward_selection_loocv(X_data, y_data, n_features_to_select=10):
    n_total_features = X_data.shape[1]
    selected = []
    remaining = list(range(n_total_features))
    best_acc = None

    for _ in range(n_features_to_select):
        best_feature = None
        best_feature_acc = -1.0

        for feature in remaining:
            candidate = selected + [feature]
            acc = loocv_accuracy_1nn(X_data, y_data, candidate)
            if (acc > best_feature_acc) or (acc == best_feature_acc and (best_feature is None or feature < best_feature)):
                best_feature_acc = acc
                best_feature = feature

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_acc = best_feature_acc

    return selected, best_acc

selected_features, loocv_acc = greedy_forward_selection_loocv(X, y, n_features_to_select=10)

print('Selected feature indices (0-based):', selected_features)
print('Selected feature indices (1-based):', [f + 1 for f in selected_features])
print(f'LOOCV accuracy using selected 10 features: {loocv_acc:.4f}')

Selected feature indices (0-based): [0, 1, 94, 65, 6, 64, 73, 55, 24, 35]
Selected feature indices (1-based): [1, 2, 95, 66, 7, 65, 74, 56, 25, 36]
LOOCV accuracy using selected 10 features: 0.8333


### Nested leave-one-out cross-validation

In [15]:
# Part 2: Nested leave-one-out cross-validation
def predict_1nn(train_X, train_y, test_x, feature_idx):
    dists = np.sum(train_X[:, feature_idx] != test_x[feature_idx], axis=1)
    nearest = np.argmin(dists)
    return train_y[nearest]

n_samples = X.shape[0]
outer_predictions = np.zeros(n_samples, dtype=int)
outer_selected_features = []

for i in range(n_samples):
    train_mask = np.ones(n_samples, dtype=bool)
    train_mask[i] = False

    X_train, y_train = X[train_mask], y[train_mask]
    X_test = X[i]

    selected_inner, _ = greedy_forward_selection_loocv(X_train, y_train, n_features_to_select=10)
    outer_selected_features.append(selected_inner)

    outer_predictions[i] = predict_1nn(X_train, y_train, X_test, selected_inner)

nested_loocv_acc = np.mean(outer_predictions == y)

print(f'Nested LOOCV accuracy (unbiased estimate): {nested_loocv_acc:.4f}')
print('First 5 outer-loop selected feature sets (0-based):')
for j in range(5):
    print(f'  Fold {j+1}:', outer_selected_features[j])

Nested LOOCV accuracy (unbiased estimate): 0.5000
First 5 outer-loop selected feature sets (0-based):
  Fold 1: [0, 1, 94, 65, 6, 64, 73, 55, 24, 35]
  Fold 2: [0, 1, 94, 65, 6, 64, 73, 55, 9, 3]
  Fold 3: [0, 1, 94, 65, 6, 55, 36, 26, 16, 17]
  Fold 4: [0, 1, 94, 65, 6, 64, 76, 51, 55, 83]
  Fold 5: [0, 1, 57, 33, 18, 72, 65, 70, 4, 61]


### Analyse the results

Analyses of results:
- As told in the exercise description, features are random Bernoulli(0.5) noise with no true signal.
- Greedy forward selection with LOOCV (single layer) still searches many feature combinations and picks the best-looking one.
- That best score is optimistically biased (winner's curse), with enough tries, some feature sets look good by chance.
- Nested LOOCV avoids this optimism by repeating feature selection inside each outer training split and evaluating on held-out samples.
- Therefore, nested LOOCV gives a more realistic estimate of generalization performance on unseen data.

Observed values:
- Single-layer LOOCV after feature selection: 0.8333, winner's curse
- Nested LOOCV estimate: 0.5000. This result is according to our dataset

### AI usage

- AI was used to help me in writing and understanding the algorithms for the Greedy forward selection with LOOCV and Nested Leave on out.